In [2]:
import pandas as pd

# Observation
'observation_concept_id' and 'observation_source_concept_id' can both contain IDs, which we want to match exactly to any of the IDs in 'ids_df'

When 'observation_concept_id' is 0, we are also interested in exact matches from 'observation_source_value' to codes in 'codes_long_df' and in codes containing 'measle' or 'mmr' but not containing 'allerg' or 'reaction'

In [ ]:
ids_df = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_id.csv')  
codes_df = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_code.csv') 
codes_long_df = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/measles_long_code.csv')  

id_list = ids_df['concept_id'].dropna().astype(int).unique().tolist()
code_list = codes_long_df['concept_code'].dropna().astype(str).unique().tolist()

# format ID list for SQL IN clause
ids_sql = ', '.join(map(str, id_list))

# format code list for SQL IN clause
codes_sql = ', '.join(f"'{code}'" for code in code_list)

# regex pattern for keyword search (case-insensitive)
regex_include = '(?i)(measle|mmr)'
regex_exclude = '(?i)(allerg|reaction)'


client = bq.Client(project='law-nero-phi-dho-scc-covid')

PROJECT_ID = "law-nero-phi-dho-scc-covid"
DATASET = "afc0125"
TABLE = "Observation"

for i in range(1990,2026):
                
    QUERY = f"""

    SELECT *
    FROM {PROJECT_ID}.{DATASET}.{TABLE} WHERE 
    EXTRACT(YEAR FROM encounterdate) = {i} AND
    observation_concept_id IN ({ids_sql}) OR
    observation_source_concept_id IN ({ids_sql}) OR
    (
        observation_concept_id = 0 AND (
            observation_source_value IN ({codes_sql}) OR
            (REGEXP_CONTAINS(observation_source_value, '{regex_include}')
             AND NOT REGEXP_CONTAINS(observation_source_value, '{regex_exclude}'))
        )
    )

    """

    query_job = client.query(QUERY)
    df = query_job.to_dataframe()

    df.to_csv(f"/share/pi/deho-pi/AFC/BQ/Observation_0125_measles/Observation_0125_{i}.csv.gz", compression = "gzip", escapechar='\\')

    del df
    del query_job


In [ ]:
# 